Data Transormation part-1

Topics to be covered:
1) Reading CSV Data
2) Understanding DataFrames
3) Common Transformations
4) Data Cleaning
5) Aggregations
6) Window Functions
7) Joins
8) Writing Transformed Data

In [17]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

In [2]:
# Create SparkSession:

spark = SparkSession.builder.appName("CSV Transformation").getOrCreate()

In [3]:
# Read CSV:
df = spark.read.csv(
    "dataFiles/employees.csv",
    header=True,
    inferSchema=True
    )


In [4]:
# View Data:
df.show()

+---+-----+----------+------+---+
| id| name|department|salary|age|
+---+-----+----------+------+---+
|  1| John|        IT| 60000| 30|
|  2|Alice|        HR| 50000| 28|
|  3|  Bob|        IT| 70000| 35|
|  4| Emma|   Finance| 80000| 40|
+---+-----+----------+------+---+



In [5]:
# Check Schema:
df.printSchema()

root
 |-- id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- age: integer (nullable = true)



In [6]:
# Selecting Specific Columns: 
df.select("name", "salary").show()

+-----+------+
| name|salary|
+-----+------+
| John| 60000|
|Alice| 50000|
|  Bob| 70000|
| Emma| 80000|
+-----+------+



In [10]:
# Other way to Select and show specific columns:
df.select(
    col("name"),
    col("salary")
).show()

+-----+------+
| name|salary|
+-----+------+
| John| 60000|
|Alice| 50000|
|  Bob| 70000|
| Emma| 80000|
+-----+------+



In [11]:
# Rename Columns:
df.select(
    col("salary").alias("annual_salary")
).show()

+-------------+
|annual_salary|
+-------------+
|        60000|
|        50000|
|        70000|
|        80000|
+-------------+



In [12]:
# Filtering Data:

# Employees with salary > 60000:

df.filter(df.salary > 60000).show()

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
|  3| Bob|        IT| 70000| 35|
|  4|Emma|   Finance| 80000| 40|
+---+----+----------+------+---+



In [13]:
# Employees with salary > 60000:
df.where("salary > 60000").show()

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
|  3| Bob|        IT| 70000| 35|
|  4|Emma|   Finance| 80000| 40|
+---+----+----------+------+---+



In [14]:
# Multiple Conditions:
# Get the Employees whose salary is above 50000 and dept is IT.

df.filter(
    (df.salary > 50000) &
    (df.department == "IT")
).show()

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
|  1|John|        IT| 60000| 30|
|  3| Bob|        IT| 70000| 35|
+---+----+----------+------+---+



In [16]:
# Adding new column: 

# Increase the salary of employees by 10% :

df = df.withColumn(
    "new_salary",
    col("salary") * 1.10
)

df.show()

+---+-----+----------+------+---+-----------------+
| id| name|department|salary|age|       new_salary|
+---+-----+----------+------+---+-----------------+
|  1| John|        IT| 60000| 30|          66000.0|
|  2|Alice|        HR| 50000| 28|55000.00000000001|
|  3|  Bob|        IT| 70000| 35|          77000.0|
|  4| Emma|   Finance| 80000| 40|          88000.0|
+---+-----+----------+------+---+-----------------+



In [20]:
# String Transformation:

df.withColumn(
    "name_upper",
    upper(col("name"))
).show()

+---+-----+----------+------+---+-----------------+----------+
| id| name|department|salary|age|       new_salary|name_upper|
+---+-----+----------+------+---+-----------------+----------+
|  1| John|        IT| 60000| 30|          66000.0|      JOHN|
|  2|Alice|        HR| 50000| 28|55000.00000000001|     ALICE|
|  3|  Bob|        IT| 70000| 35|          77000.0|       BOB|
|  4| Emma|   Finance| 80000| 40|          88000.0|      EMMA|
+---+-----+----------+------+---+-----------------+----------+



In [19]:
df.withColumn(
    "name_lower",
    lower(col("name"))
).show()

+---+-----+----------+------+---+-----------------+----------+
| id| name|department|salary|age|       new_salary|name_lower|
+---+-----+----------+------+---+-----------------+----------+
|  1| John|        IT| 60000| 30|          66000.0|      john|
|  2|Alice|        HR| 50000| 28|55000.00000000001|     alice|
|  3|  Bob|        IT| 70000| 35|          77000.0|       bob|
|  4| Emma|   Finance| 80000| 40|          88000.0|      emma|
+---+-----+----------+------+---+-----------------+----------+



In [21]:
# Trim Spaces:
df.withColumn(
    "name_trimmed",
    trim(col("name"))
).show()

+---+-----+----------+------+---+-----------------+------------+
| id| name|department|salary|age|       new_salary|name_trimmed|
+---+-----+----------+------+---+-----------------+------------+
|  1| John|        IT| 60000| 30|          66000.0|        John|
|  2|Alice|        HR| 50000| 28|55000.00000000001|       Alice|
|  3|  Bob|        IT| 70000| 35|          77000.0|         Bob|
|  4| Emma|   Finance| 80000| 40|          88000.0|        Emma|
+---+-----+----------+------+---+-----------------+------------+



In [22]:
df.withColumn(
    "emp_info",
    concat(col("name"), lit("-"), col("department"))
).show()

+---+-----+----------+------+---+-----------------+------------+
| id| name|department|salary|age|       new_salary|    emp_info|
+---+-----+----------+------+---+-----------------+------------+
|  1| John|        IT| 60000| 30|          66000.0|     John-IT|
|  2|Alice|        HR| 50000| 28|55000.00000000001|    Alice-HR|
|  3|  Bob|        IT| 70000| 35|          77000.0|      Bob-IT|
|  4| Emma|   Finance| 80000| 40|          88000.0|Emma-Finance|
+---+-----+----------+------+---+-----------------+------------+



In [27]:
spark.stop()

In [30]:
spark = SparkSession.builder.appName("csv_transformation2").getOrCreate()

df1 = spark.read.csv(
    "dataFiles/employees.csv",
    header=True,
    inferSchema=True
)

df1.show()

+---+-----+----------+------+----+
| id| name|department|salary| age|
+---+-----+----------+------+----+
|  1| NULL|        IT| 60000|NULL|
|  2|Alice|      NULL| 50000|  28|
|  3|  Bob|        IT| 70000|  35|
|  4| Emma|   Finance| 80000|  40|
|  5| NULL|     Sales| 55000|  29|
|  6|David|      NULL| 75000|  32|
+---+-----+----------+------+----+



In [ ]:
# Check Nulls:

df1.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df1.columns
]).show()``

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
|  0|   2|         2|     0|  1|
+---+----+----------+------+---+



In [36]:
# Drop null rows:
df1.na.drop().show()

+---+----+----------+------+---+
| id|name|department|salary|age|
+---+----+----------+------+---+
|  3| Bob|        IT| 70000| 35|
|  4|Emma|   Finance| 80000| 40|
+---+----+----------+------+---+



In [41]:
df1.na.fill({
    "name": "Unkown",
    "department": "Unkown",
    "age": 0
}).show()

+---+------+----------+------+---+
| id|  name|department|salary|age|
+---+------+----------+------+---+
|  1|Unkown|        IT| 60000|  0|
|  2| Alice|    Unkown| 50000| 28|
|  3|   Bob|        IT| 70000| 35|
|  4|  Emma|   Finance| 80000| 40|
|  5|Unkown|     Sales| 55000| 29|
|  6| David|    Unkown| 75000| 32|
+---+------+----------+------+---+

